# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE

In [2]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 


## 1. Distribution of the target variable

In [3]:
fraud["fraud"].value_counts()

fraud
0.0    912597
1.0     87403
Name: count, dtype: int64

In [4]:
fraud["fraud"].value_counts(normalize=True)

fraud
0.0    0.912597
1.0    0.087403
Name: proportion, dtype: float64

Solo el **8.7%** de las transacciones son fraudulentas frente al **91.3%** que son legítimas. La proporción está muy lejos de un 50/50, por lo que claramente estamos ante un **dataset desbalanceado**. Esto es importante porque un modelo entrenado sin tener esto en cuenta tenderá a predecir casi siempre la clase mayoritaria (legítima) y a fallar en detectar la clase minoritaria (fraude), que suele ser la que realmente nos interesa identificar.

## 2. Train a Logistic Regression

In [5]:
X = fraud.drop("fraud", axis=1)
y = fraud["fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape

((800000, 7), (200000, 7))

In [6]:
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

pred = log_reg.predict(X_test)

## 3. Evaluate the model

Dado que las clases están desbalanceadas, el **accuracy** no es una buena métrica: un modelo que siempre prediga "no fraude" tendría ~91% de accuracy sin ser útil. Como nos interesa especialmente detectar la clase minoritaria (fraude = 1), es mejor fijarnos en **precision, recall y f1-score de la clase 1**, en la **matriz de confusión** y en el **ROC AUC** (que no se ve afectado por el desbalanceo).

In [7]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

         0.0       0.96      0.99      0.98    182557
         1.0       0.89      0.60      0.72     17443

    accuracy                           0.96    200000
   macro avg       0.93      0.80      0.85    200000
weighted avg       0.96      0.96      0.95    200000



In [8]:
confusion_matrix(y_test, pred)

array([[181279,   1278],
       [  6993,  10450]])

In [9]:
roc_auc_score(y_test, log_reg.predict_proba(X_test)[:, 1])

0.9669528148268153

**Observación:** el modelo base obtiene un recall de la clase fraude (1) en torno al **0.60**, es decir, solo detecta el 60% de los fraudes reales, aunque su precision es alta (~0.89). El f1-score de la clase fraude es de aproximadamente **0.72**. El ROC AUC (~0.97) es bastante bueno, pero el recall bajo en la clase de interés confirma el efecto del desbalanceo: el modelo es conservador a la hora de predecir fraude.

## 4. Oversample

Vamos a balancear el conjunto de entrenamiento duplicando (con reemplazo) observaciones de la clase minoritaria (fraude) hasta igualar el número de observaciones de la clase mayoritaria. **Importante:** el oversampling se aplica únicamente sobre el conjunto de *train*, nunca sobre el *test*, para evitar fuga de información (data leakage).

In [10]:
train = pd.concat([X_train, y_train], axis=1)

legit = train[train["fraud"] == 0]
fraud_cls = train[train["fraud"] == 1]

fraud_over = fraud_cls.sample(len(legit), replace=True, random_state=42)

train_over = pd.concat([legit, fraud_over])

X_train_over = train_over.drop("fraud", axis=1)
y_train_over = train_over["fraud"]

y_train_over.value_counts()

fraud
0.0    730040
1.0    730040
Name: count, dtype: int64

In [11]:
log_reg_over = LogisticRegression(max_iter=1000)
log_reg_over.fit(X_train_over, y_train_over)

pred_over = log_reg_over.predict(X_test)

print(classification_report(y_test, pred_over))

              precision    recall  f1-score   support

         0.0       1.00      0.93      0.96    182557
         1.0       0.58      0.95      0.72     17443

    accuracy                           0.93    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



In [12]:
confusion_matrix(y_test, pred_over)

array([[170354,  12203],
       [   851,  16592]])

In [13]:
roc_auc_score(y_test, log_reg_over.predict_proba(X_test)[:, 1])

0.9795512900022268

**¿Mejora el rendimiento?** Sí, claramente para la clase de interés (fraude). El recall de la clase 1 sube desde ~0.60 hasta ~0.95, y el ROC AUC también mejora ligeramente (de ~0.97 a ~0.98). A cambio, la precision de la clase 1 baja (de ~0.89 a ~0.58), lo que significa que ahora hay más falsos positivos. El f1-score de la clase fraude se mantiene similar (~0.72), pero ahora estamos detectando muchos más fraudes reales, lo cual suele ser preferible en un caso de uso de detección de fraude, donde el coste de un fraude no detectado (falso negativo) suele ser mayor que el de revisar una transacción legítima marcada como sospechosa (falso positivo).

## 5. Undersample

Ahora balanceamos reduciendo la clase mayoritaria (legítimas) hasta igualar el número de observaciones de la clase minoritaria (fraude), de nuevo solo en el conjunto de *train*.

In [14]:
legit_under = legit.sample(len(fraud_cls), random_state=42)
fraud_under = fraud_cls

train_under = pd.concat([legit_under, fraud_under])

X_train_under = train_under.drop("fraud", axis=1)
y_train_under = train_under["fraud"]

y_train_under.value_counts()

fraud
0.0    69960
1.0    69960
Name: count, dtype: int64

In [15]:
log_reg_under = LogisticRegression(max_iter=1000)
log_reg_under.fit(X_train_under, y_train_under)

pred_under = log_reg_under.predict(X_test)

print(classification_report(y_test, pred_under))

              precision    recall  f1-score   support

         0.0       1.00      0.93      0.96    182557
         1.0       0.58      0.95      0.72     17443

    accuracy                           0.93    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



In [16]:
confusion_matrix(y_test, pred_under)

array([[170293,  12264],
       [   840,  16603]])

In [17]:
roc_auc_score(y_test, log_reg_under.predict_proba(X_test)[:, 1])

0.9795377617746155

**¿Mejora el rendimiento?** Los resultados con undersampling son prácticamente idénticos a los de oversampling: recall de la clase fraude en torno a ~0.95, precision ~0.58, f1-score ~0.72 y ROC AUC ~0.98. Es decir, también mejora notablemente la detección de fraude respecto al modelo base, con la ventaja añadida de que el entrenamiento es mucho más rápido al usar muchas menos filas (~140.000 frente a ~1.460.000 en el caso del oversample), aunque con el riesgo de estar descartando información útil de la clase mayoritaria al eliminar tantas observaciones.

## 6. SMOTE

Por último, usamos **SMOTE** (Synthetic Minority Oversampling Technique), que en lugar de duplicar observaciones existentes de la clase minoritaria, genera nuevas observaciones sintéticas interpolando entre vecinos cercanos de esa clase.

In [18]:
sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

y_train_sm.value_counts()

fraud
0.0    730040
1.0    730040
Name: count, dtype: int64

In [19]:
log_reg_sm = LogisticRegression(max_iter=1000)
log_reg_sm.fit(X_train_sm, y_train_sm)

pred_sm = log_reg_sm.predict(X_test)

print(classification_report(y_test, pred_sm))

              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182557
         1.0       0.58      0.95      0.72     17443

    accuracy                           0.94    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.94      0.94    200000



In [20]:
confusion_matrix(y_test, pred_sm)

array([[170443,  12114],
       [   872,  16571]])

In [21]:
roc_auc_score(y_test, log_reg_sm.predict_proba(X_test)[:, 1])

0.9794669738323574

**¿Mejora el rendimiento?** Sí, SMOTE da resultados muy similares a oversample/undersample: recall de la clase fraude ~0.95, precision ~0.58, f1-score ~0.72 y ROC AUC ~0.98, frente al recall ~0.60 y f1 ~0.72 (con mucha menor sensibilidad) del modelo base.

### Conclusión final

Las tres técnicas de balanceo (Oversample, Undersample y SMOTE) mejoran sustancialmente el **recall** de la clase fraude respecto al modelo base entrenado con datos desbalanceados, pasando de detectar ~60% de los fraudes a detectar ~95%. El ROC AUC también mejora ligeramente en los tres casos (de ~0.967 a ~0.979-0.980). La precision de la clase fraude baja en los tres casos balanceados, generando más falsos positivos, pero en un problema de detección de fraude esto suele ser un compromiso aceptable, ya que el coste de no detectar un fraude (falso negativo) suele ser mucho mayor que el de revisar manualmente una transacción legítima marcada como sospechosa (falso positivo). Las tres técnicas de balanceo se comportan de forma muy parecida entre sí en este caso.